# 05 — GPT-5 Results and Reports

Loads all GPT-5 evaluation results from checkpoint files and produces
summary tables, domain reports, and saves final results.
No API calls, no model loading — runs in seconds.

## Prerequisites
Run `04_eval_gpt5.ipynb` first to generate checkpoint files in `paper2/results/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

ROOT        = '/content/drive/MyDrive/paper_codes/paper2'
RESULTS_DIR = f'{ROOT}/results'

DOMAINS    = ['blocksworld', 'blocksworld_3', 'mystery', 'mystery_3', 'logistics']
DOM_SHORT  = {'blocksworld':'bw', 'blocksworld_3':'bw3', 'mystery':'mys',
              'mystery_3':'mys3', 'logistics':'log'}
ALL_CONDS  = ['G5_direct', 'G5_stepwise']

print(f'RESULTS_DIR: {RESULTS_DIR}')

# List available checkpoint files
ckpts = [f for f in os.listdir(RESULTS_DIR) if f.startswith('ckpt_G5')]
print(f'Checkpoint files found: {sorted(ckpts)}')

## 1. Load Results from Checkpoints

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def load_direct_from_checkpoint(domain):
    """
    Direct checkpoint stores: plan_id, label ('A'/'B'), step (predicted failing step or None).
    plan_gold comes from the direct test records (label_int field).
    """
    ckpt_path = os.path.join(RESULTS_DIR, f'ckpt_G5_direct_{domain}.json')
    di_path   = os.path.join(ROOT, 'data', 'planbench_direct_test.jsonl')
    if not os.path.exists(ckpt_path):
        print(f'  MISSING: {ckpt_path}'); return None

    ckpt     = json.load(open(ckpt_path))
    di_recs  = {json.loads(l)['plan_id']: json.loads(l)
                for l in open(di_path)
                if json.loads(l)['domain'] == domain}

    print(f'  G5_direct|{domain}: {len(ckpt)} plans from checkpoint')

    plan_golds, plan_preds = [], []
    fail_step_golds, fail_step_preds = [], []

    for r in ckpt:
        pid      = r['plan_id']
        label    = r.get('label', 'B')
        pred_step = r.get('step')
        if label == 'EMPTY': label = 'B'  # failed API call treated as B

        di_rec   = di_recs.get(pid)
        if di_rec is None: continue
        plan_gold = di_rec['label_int']          # 0=invalid 1=valid
        plan_pred = 1 if label == 'A' else 0

        plan_golds.append(plan_gold)
        plan_preds.append(plan_pred)

        # Failing step: only for invalid plans where model predicted invalid
        if plan_gold == 0 and pred_step is not None and pred_step > 0:
            # gold failing step from di_rec fail_reason
            fail_reason = di_rec.get('fail_reason', '')
            # We don't have gold_fs in direct records easily —
            # use sw records via sw_test path
            pass  # skip failing step for direct (not reported in paper)

    acc_plan = accuracy_score(plan_golds, plan_preds)
    f1_plan  = f1_score(plan_golds, plan_preds, average='macro', zero_division=0)
    cm_plan  = confusion_matrix(plan_golds, plan_preds, labels=[0, 1])

    return {
        'cond': 'G5_direct', 'domain': domain, 'eval_type': 'direct',
        'n_plans' : len(plan_golds),
        'n_actions': 0,
        'plan_acc': round(acc_plan, 4),
        'plan_f1' : round(f1_plan,  4),
        'plan_cm' : cm_plan.tolist(),
        'action_acc': None, 'action_f1': None, 'action_cm': None,
        'failing_step_acc': None, 'n_failing_step_evaluated': 0,
        'n_errors': sum(g != p for g, p in zip(plan_golds, plan_preds)),
        'plan_golds': plan_golds, 'plan_preds': plan_preds,
        'action_golds': [], 'action_preds': [],
        'fail_step_golds': [], 'fail_step_preds': [],
    }


def load_stepwise_from_checkpoint(domain):
    """
    Stepwise checkpoint stores: plan_id, plan_pred, plan_gold, action_preds, action_golds.
    """
    ckpt_path = os.path.join(RESULTS_DIR, f'ckpt_G5_stepwise_{domain}.json')
    if not os.path.exists(ckpt_path):
        print(f'  MISSING: {ckpt_path}'); return None

    ckpt = json.load(open(ckpt_path))
    print(f'  G5_stepwise|{domain}: {len(ckpt)} plans from checkpoint')

    plan_golds     = [r['plan_gold'] for r in ckpt]
    plan_preds_out = [r['plan_pred'] for r in ckpt]
    action_golds   = [g for r in ckpt for g in r.get('action_golds', [])]
    action_preds   = [p for r in ckpt for p in r.get('action_preds', [])]

    # Failing step
    fail_step_golds, fail_step_preds = [], []
    for r in ckpt:
        if r['plan_gold'] == 0:
            ag = r.get('action_golds', [])
            ap = r.get('action_preds', [])
            gold_fs = next((i+1 for i, g in enumerate(ag) if g == 0), None)
            pred_fs = next((i+1 for i, p in enumerate(ap) if p == 0), None)
            if gold_fs is not None and pred_fs is not None:
                fail_step_golds.append(gold_fs)
                fail_step_preds.append(pred_fs)

    acc_plan = accuracy_score(plan_golds, plan_preds_out)
    f1_plan  = f1_score(plan_golds, plan_preds_out, average='macro', zero_division=0)
    cm_plan  = confusion_matrix(plan_golds, plan_preds_out, labels=[0, 1])
    acc_act  = accuracy_score(action_golds, action_preds) if action_golds else None
    f1_act   = f1_score(action_golds, action_preds, average='macro', zero_division=0) if action_golds else None
    cm_act   = confusion_matrix(action_golds, action_preds, labels=[0, 1]) if action_golds else None
    fs_acc   = (sum(g == p for g, p in zip(fail_step_golds, fail_step_preds))
                / max(1, len(fail_step_golds))) if fail_step_golds else None

    return {
        'cond': 'G5_stepwise', 'domain': domain, 'eval_type': 'stepwise',
        'n_plans'   : len(plan_golds),
        'n_actions' : len(action_golds),
        'plan_acc'  : round(acc_plan, 4),
        'plan_f1'   : round(f1_plan,  4),
        'plan_cm'   : cm_plan.tolist(),
        'action_acc': round(acc_act, 4) if acc_act is not None else None,
        'action_f1' : round(f1_act,  4) if f1_act  is not None else None,
        'action_cm' : cm_act.tolist()    if cm_act  is not None else None,
        'failing_step_acc': round(fs_acc, 4) if fs_acc is not None else None,
        'n_failing_step_evaluated': len(fail_step_golds),
        'n_errors'  : sum(g != p for g, p in zip(plan_golds, plan_preds_out)),
        'plan_golds': plan_golds, 'plan_preds': plan_preds_out,
        'action_golds': action_golds, 'action_preds': action_preds,
        'fail_step_golds': fail_step_golds, 'fail_step_preds': fail_step_preds,
    }


# Load all results
all_results = {'G5_direct': {}, 'G5_stepwise': {}}

for domain in DOMAINS:
    r = load_direct_from_checkpoint(domain)
    if r: all_results['G5_direct'][domain] = r

for domain in DOMAINS:
    r = load_stepwise_from_checkpoint(domain)
    if r: all_results['G5_stepwise'][domain] = r

print()
for cond in ALL_CONDS:
    print(f'{cond}: {list(all_results[cond].keys())}')

## 2. Summary Tables

In [ ]:
SEP  = '─' * 82
SEP2 = '=' * 82

def summary_table(metric, eval_type, label):
    conds = [c for c in ALL_CONDS
             if any(all_results.get(c, {}).get(d, {}).get('eval_type') == eval_type
                    for d in DOMAINS)]
    if not conds: return
    print(f'\n{SEP}\n{label}')
    print(f'{"Condition":<25}  ' + '  '.join(f'{DOM_SHORT[d]:>6}' for d in DOMAINS) + f'  {"Mean":>6}')
    print(SEP)
    for cond in conds:
        vals = [all_results[cond].get(d, {}).get(metric) for d in DOMAINS]
        valid = [v for v in vals if v is not None]
        mean  = np.mean(valid) * 100 if valid else float('nan')
        row   = '  '.join(f'{v*100:6.1f}' if v is not None else f'  {"N/A":>4}' for v in vals)
        print(f'{cond:<25}  {row}  {mean:6.1f}')

print(SEP2)
print('GPT-5 EVALUATION RESULTS')
print(SEP2)

summary_table('plan_f1',          'direct',   'PLAN F1-MACRO (%) — Direct')
summary_table('plan_f1',          'stepwise', 'PLAN F1-MACRO (%) — Stepwise')
summary_table('action_f1',        'stepwise', 'ACTION F1-MACRO (%) — Stepwise')
summary_table('failing_step_acc', 'stepwise', 'FAILING STEP ACCURACY (%) — Stepwise')
summary_table('plan_acc',         'direct',   'PLAN ACCURACY (%) — Direct')
summary_table('plan_acc',         'stepwise', 'PLAN ACCURACY (%) — Stepwise')

## 3. Per-Domain Reports

In [ ]:
for cond in ALL_CONDS:
    for domain in DOMAINS:
        r = all_results[cond].get(domain)
        if not r: continue
        print(f'\n{"="*65}')
        print(f'{cond} | {domain.upper()} | {r["eval_type"].upper()}')
        print(f'{"="*65}')
        print(f'Plans: {r["n_plans"]}  |  '
              f'plan_acc={r["plan_acc"]:.4f}  plan_f1={r["plan_f1"]:.4f}')
        if r.get('action_f1') is not None:
            print(f'Actions: {r["n_actions"]}  |  '
                  f'action_acc={r["action_acc"]:.4f}  action_f1={r["action_f1"]:.4f}')
        if r.get('failing_step_acc') is not None:
            print(f'Failing step acc={r["failing_step_acc"]:.4f}  '
                  f'(evaluated on {r["n_failing_step_evaluated"]} plans)')
        # Confusion matrix
        cm = r['plan_cm']
        tn, fp, fn, tp = cm[0][0], cm[0][1], cm[1][0], cm[1][1]
        print(f'Plan CM:  TN={tn}  FP={fp}  FN={fn}  TP={tp}')
        print(classification_report(r['plan_golds'], r['plan_preds'],
              target_names=['Invalid', 'Valid'], zero_division=0))

## 4. Save Results

In [ ]:
# Save metrics (no raw arrays)
SKIP_KEYS = {'plan_golds','plan_preds','action_golds','action_preds',
             'fail_step_golds','fail_step_preds'}
save_dict = {
    cond: {domain: {k:v for k,v in r.items() if k not in SKIP_KEYS}
           for domain, r in dom_res.items()}
    for cond, dom_res in all_results.items()
}
out_path = os.path.join(RESULTS_DIR, 'results_gpt5.json')
with open(out_path, 'w') as f:
    json.dump(save_dict, f, indent=2)
print(f'Results saved -> {out_path}')

# Save raw predictions
RAW_KEYS = {'plan_golds','plan_preds','action_golds','action_preds',
            'fail_step_golds','fail_step_preds'}
raw_dict = {
    cond: {domain: {k:v for k,v in r.items() if k in RAW_KEYS}
           for domain, r in dom_res.items()}
    for cond, dom_res in all_results.items()
}
raw_path = os.path.join(RESULTS_DIR, 'raw_preds_gpt5.json')
with open(raw_path, 'w') as f:
    json.dump(raw_dict, f)
print(f'Raw predictions saved -> {raw_path}')
print()
print('=== DONE ==='  )